# Tome C3, Chapter 7: Incompleteness (Python companion)

Exercises first, solutions at the end. We code formulas by numbers, code sequences with Gödel's β-function, check $\Pi_1$ statements by search, and build a self-referential sentence mechanically, then check that it refers to itself. In the second half we run Goodstein sequences with their ordinals, compute exactly when the sequence of $4$ dies, fight hydras, and measure how compressible strings are.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools, math, random, time, zlib
from math import gcd, factorial
import matplotlib.pyplot as plt
random.seed(7)

## Exercises

**P7.1 (★) Gödel numbers.** Code the symbols of arithmetic by small numbers and a formula by the product $2^{s_1+1}3^{s_2+1}\cdots p_k^{s_k+1}$. Encode and decode the axioms of $\mathrm Q$ (Definition 7.1), and compare the size of these codes with a second numbering, which reads the text of the formula as a number in base $256$.

**P7.2 (★) The β-function.** Implement Gödel's construction (Lemma 7.5) with the Chinese remainder theorem, check it on random sequences, and find by search the smallest pairs $(c,d)$ coding short sequences (Exercise 7.2). How large is Gödel's $c$ compared with the smallest one?

**P7.3 (★) $\Pi_1$ statements are refutable by computation.** Goldbach's conjecture is $\Pi_1$ (Section 7.1). Check it up to $10^5$ and plot the number of ways of writing $2n$ as a sum of two primes. Explain why a $\Pi_1$ sentence that is undecided in $\mathrm{PA}$ (or in any sound theory containing $\mathrm Q$) must be true.

**P7.4 (★★) A sentence about itself.** With the base-$256$ numbering of P7.1, implement the function $\mathrm{diag}$ of Lemma 7.13 on texts, build the sentence $\delta$ for $\varphi(y)=$ ``$y$ is not the code of a provable sentence'', and check that the number $\delta$ talks about is its own code.

**P7.5 (★) Goodstein sequences and their ordinals.** Implement hereditary base-$b$ notation, Goodstein sequences, the map $o_b$ to Cantor normal forms and their order. Check on the sequences of $1,\dots,6$ that the ordinals decrease (Theorem 7.20), and reproduce the table of Section 7.5.

**P7.6 (★★) When does the sequence of 4 die?** Show that consuming a term $\omega$ at base $b$ takes the base to $2b+1$, and consuming $\omega^2$ takes it to $2^{b+1}(b+1)-1$; check these rules by brute force for small bases. Deduce the base, $3\cdot2^{402653211}-1$, at which the sequence of $4$ reaches $0$, and the number of digits of the number of steps.

**P7.7 (★★) Hercules and the hydra.** Simulate the hydra game of Figure 7.2 with random cuts, compute the CNF of Exercise 7.10 at each step and check that it decreases. How long do small hydras survive?

**P7.8 (★) Incompressible strings.** Use `zlib` as a (bad but honest) upper bound for Kolmogorov complexity. Compress random strings, strings with patterns, and the digits of $\pi$-like sequences; compare with the counting bound of Exercise 7.11.

## Solutions

### P7.1

In [ ]:
SYMBOLS = ['0', 'S', '+', '*', '=', '¬', '→', '∀', '(', ')', ','] + [f"v{i}" for i in range(10)]
NUM = {s: i for i, s in enumerate(SYMBOLS)}

def primes():
    n = 2
    while True:
        if all(n % p for p in range(2, int(n ** 0.5) + 1)):
            yield n
        n += 1

def tokenize(text):
    out, i = [], 0
    while i < len(text):
        if text[i] == ' ':
            i += 1; continue
        if text[i] == 'v':
            j = i + 1
            while j < len(text) and text[j].isdigit(): j += 1
            out.append(text[i:j]); i = j
        else:
            out.append(text[i]); i += 1
    return out

def godel(text):
    g = 1
    for p, s in zip(primes(), tokenize(text)):
        g *= p ** (NUM[s] + 1)
    return g

def ungodel(g):
    out = []
    for p in primes():
        if g == 1: break
        e = 0
        while g % p == 0:
            g //= p; e += 1
        out.append(SYMBOLS[e - 1])
    return ' '.join(out)

def text_code(text):
    return int.from_bytes(text.encode('utf-8'), 'big')

Q = ["∀v0 ¬(S(v0)=0)",
     "∀v0 ∀v1 (S(v0)=S(v1) → v0=v1)",
     "∀v0 (¬(v0=0) → ¬∀v1 ¬(v0=S(v1)))",
     "∀v0 (+(v0,0)=v0)",
     "∀v0 ∀v1 (+(v0,S(v1))=S(+(v0,v1)))",
     "∀v0 (*(v0,0)=0)",
     "∀v0 ∀v1 (*(v0,S(v1))=+(*(v0,v1),v0))"]
for ax in Q:
    g = godel(ax)
    assert ungodel(g).replace(' ', '') == ''.join(tokenize(ax))
    print(f"{ax:42s} prime-power code has {len(str(g)):4d} digits; text code has {len(str(text_code(ax))):3d} digits")

Both numberings are injective and computable in both directions, which is all the theory needs; the prime-power codes are much larger (their exponents carry the symbols), which is why practical work, and P7.4, prefers the text code. Note how $\exists$ is written: $\exists y\,\varphi$ is $\neg\forall y\,\neg\varphi$, as in Chapter 6.

### P7.2

In [ ]:
def beta(c, d, i):
    return c % (1 + (i + 1) * d)

def crt(residues, moduli):
    x, M = 0, 1
    for r, m in zip(residues, moduli):
        # solve x + M*t = r (mod m)
        t = ((r - x) * pow(M, -1, m)) % m
        x, M = x + M * t, M * m
    return x

def godel_beta(seq):
    n = len(seq) - 1
    s = max([n] + list(seq)) + 1
    d = factorial(s)
    moduli = [1 + (i + 1) * d for i in range(n + 1)]
    return crt(seq, moduli), d

for _ in range(200):
    seq = [random.randrange(20) for _ in range(random.randrange(1, 8))]
    c, d = godel_beta(seq)
    assert [beta(c, d, i) for i in range(len(seq))] == seq
print("Gödel's construction works on 200 random sequences")

def crt_general(residues, moduli):
    # least x >= 0 with x = r_i (mod m_i) for all i, moduli not necessarily coprime; None if inconsistent
    x, M = 0, 1
    for r, m in zip(residues, moduli):
        g = gcd(M, m)
        if (r - x) % g:
            return None
        t = ((r - x) // g * pow(M // g, -1, m // g)) % (m // g) if m // g > 1 else 0
        x, M = x + M * t, M * m // g
        x %= M
    return x

def smallest_pair(seq, dmax=2000):
    best = None
    for d in range(1, dmax):
        moduli = [1 + (i + 1) * d for i in range(len(seq))]
        if any(a >= m for a, m in zip(seq, moduli)):
            continue
        c = crt_general(seq, moduli)
        if c is not None and (best is None or c < best[0]):
            best = (c, d)
    return best

for seq in [[2, 0, 1], [1, 1, 2, 3, 5], [3, 1, 4, 1, 5]]:
    c, d = godel_beta(seq)
    print(f"{seq}: Gödel's pair has c with {len(str(c))} digits (d = {d}); smallest pair by search: {smallest_pair(seq)}")

For $(2,0,1)$, Gödel's recipe gives $c=1521$ and $d=6$ as in the solution of Exercise 7.2, while $(27,4)$ already works (moduli $5,9,13$, not even pairwise coprime: coprimality is sufficient, not necessary). For each $d$ the least $c$ comes from the Chinese remainder theorem for non-coprime moduli. The recipe is wasteful (the factorial makes $d$ huge, and $c$ is about the product of the moduli) but it is a proof: it always works, and the theory only needs existence.

### P7.3

In [ ]:
N = 20000
sieve = bytearray([1]) * (N + 1); sieve[0] = sieve[1] = 0
for i in range(2, int(N ** 0.5) + 1):
    if sieve[i]:
        sieve[i * i::i] = bytearray(len(range(i * i, N + 1, i)))
primes_list = [i for i in range(N + 1) if sieve[i]]
t0 = time.time()
counts = []
for n in range(4, N + 1, 2):
    c = 0
    for p in primes_list:
        if p > n // 2: break
        if sieve[n - p]: c += 1
    if c == 0:
        print("counterexample!", n)
    counts.append(c)
print(f"Goldbach holds for every even number from 4 to {N} ({time.time() - t0:.1f} s)")
plt.figure(figsize=(6, 3))
plt.plot(range(4, N + 1, 2), counts, ',', alpha=0.5)
plt.xlabel("2n"); plt.ylabel("number of ways"); plt.title("Goldbach's comet"); plt.tight_layout(); plt.show()

A false $\Pi_1$ sentence $\forall y\,\delta(y)$ has a counterexample $m$, and $\neg\delta(\overline m)$ is a true $\Delta_0$ sentence, provable in $\mathrm Q$ (Theorem 7.4); so the sentence is refutable. Hence a $\Pi_1$ sentence that $\mathrm{PA}$ does not refute is true, and one that $\mathrm{PA}$ cannot decide at all, like $G_{\mathrm{PA}}$ or $\mathrm{Con}_{\mathrm{PA}}$, is true. If Goldbach's conjecture were independent of $\mathrm{PA}$, it would be true. Checking up to $2\cdot10^4$ takes a moment; Oliveira e Silva, Herzog and Pardi checked it up to $4\cdot10^{18}$ in 2014. The plot, the ``comet'', shows the number of representations growing roughly like $n/\log^2 n$; nothing in it is a proof.

### P7.4

We work with texts, coded in base $256$. A formula with free variable `x` is a text; $\mathrm{diag}$ replaces `x` by the numeral of the formula's own code (we write numerals in decimal for readability: in the official language $\overline n$ would be $S(S(\cdots0))$, and the argument is unchanged).

In [ ]:
def decode(n):
    return n.to_bytes((n.bit_length() + 7) // 8, 'big').decode('utf-8')

def diag(n):
    # the code of chi(numeral of n), where n is the code of chi(x)
    chi = decode(n)
    return text_code(chi.replace('x', str(n)))

phi = "¬Prov(y)"                                  # φ(y): y is not the code of a provable sentence
chi = f"∃y (D(x,y) ∧ {phi})"                      # χ(x) = ∃y (D(x,y) ∧ φ(y)), D represents diag
code_chi = text_code(chi)
delta = chi.replace('x', str(code_chi))           # δ = χ(numeral of ⌜χ⌝)
print("δ =", delta[:60] + "..." + delta[-25:])
print("length of δ:", len(delta), "characters")
print("δ says: the number y with D(", str(code_chi)[:12] + "...", ", y) is not the code of a provable sentence")
print("diag(⌜χ⌝) == ⌜δ⌝ ?", diag(code_chi) == text_code(delta))
print("does δ contain its own code as a numeral?", str(text_code(delta)) in delta)

The sentence $\delta$ mentions the code of $\chi$, not its own code, and its own code does not appear in it (it could not: the code of a text is larger than any number written in it). It says ``the result of applying $\mathrm{diag}$ to this number is unprovable'', and the result of that computation is exactly its own code. This is a quine: the program `s='...'; print(s % s)` does not contain itself either, only instructions which rebuild it.

### P7.5

In [ ]:
# CNF: tuple of (exponent CNF, coefficient) pairs, exponents strictly decreasing; 0 is ().
def cnf_cmp(a, b):
    for (ea, ca), (eb, cb) in zip(a, b):
        c = cnf_cmp(ea, eb)
        if c: return c
        if ca != cb: return -1 if ca < cb else 1
    return (len(a) > len(b)) - (len(a) < len(b))

def o(m, b):
    terms, e = [], 0
    while m:
        m, d = divmod(m, b)
        if d: terms.append((o(e, b), d))
        e += 1
    return tuple(reversed(terms))

def value(alpha, b):
    return sum(c * b ** value(e, b) for e, c in alpha)

def show(alpha):
    if not alpha: return "0"
    parts = []
    for e, c in alpha:
        if not e: parts.append(str(c)); continue
        se = show(e)
        base = "ω" if se == "1" else (f"ω^{se}" if " " not in se else f"ω^({se})")
        parts.append(base + (f"·{c}" if c > 1 else ""))
    return " + ".join(parts)

def goodstein(m, steps):
    b, out = 2, []
    while m and len(out) < steps:
        out.append((b, m, o(m, b)))
        m = value(o(m, b), b + 1) - 1
        b += 1
    out.append((b, m, o(m, b)))
    return out

for b, m, a in goodstein(4, 5):
    print(f"base {b}: {m:4d}   {show(a)}")
for m0 in range(1, 7):
    seq = goodstein(m0, 300)
    ok = all(cnf_cmp(seq[i + 1][2], seq[i][2]) < 0 for i in range(len(seq) - 1))
    print(f"m = {m0}: {len(seq) - 1} steps computed, ordinals strictly decreasing: {ok}, last term {seq[-1][1] if seq[-1][1] < 10**12 else '≈10^' + str(len(str(seq[-1][1])) - 1)}")

The ordinals decrease at every step, while the terms grow for all $m\ge4$ in the window we can afford. A CNF here is written with $\omega^{(\dots)}$ for exponents and $\omega$ for $\omega^1$; for example the start of $4$ is $\omega^\omega$.

### P7.6

At base $b$, a value whose ordinal ends in $+k$ (a finite part) loses $1$ per step, so $k$ steps take the base to $b+k$. A last term $\omega$, at base $b$: one step bumps $b$ to $b+1$ and subtracts $1$ from $(b+1)^1$, leaving the finite part $b$; then $b$ more steps: the base is $2b+1$. A last term $\omega^2$: one step leaves $(b+1)^2-1=b(b+1)+b$, ordinal $\omega\cdot b+b$; the finite part takes the base to $2b+1$, and $b$ terms $\omega$ double it $b$ times: $b'\mapsto2b'+1$ applied $b$ times to $2b+1$ gives $2^b(2b+2)-1=2^{b+1}(b+1)-1$.

In [ ]:
def run_until(alpha_target, m, b):
    # brute force: iterate from value m at base b until the ordinal equals alpha_target; return the base
    while cnf_cmp(o(m, b), alpha_target) != 0:
        m = value(o(m, b), b + 1) - 1
        b += 1
    return b

for b in range(3, 8):     # for b = 2, the number b^2 = 2^2 is ω^ω, not ω^2: that is the sequence of 4!
    assert run_until((), b, b) == 2 * b + 1                      # value b = ω at base b, consumed
    assert run_until((), b * b, b) == 2 ** (b + 1) * (b + 1) - 1  # value b^2 = ω^2 at base b, consumed
print("rules checked by brute force for b = 3..7: ω: b -> 2b+1,  ω^2: b -> 2^(b+1)(b+1) - 1")

# The sequence of 4: base 2, ordinal ω^ω; one step gives base 3, ordinal ω^2·2 + ω·2 + 2.
b = 3
b += 2                       # the finite part 2
for _ in range(2):           # ω·2
    b = 2 * b + 1
print("after ω^2·2 + ω·2 + 2 → ω^2·2: base", b)
b = 2 ** (b + 1) * (b + 1) - 1          # first ω^2
print("after the first ω^2: base", b, "= 3·2^27 - 1 ?", b == 3 * 2 ** 27 - 1)
# second ω^2: b -> 2^(b+1) (b+1) - 1 with b + 1 = 3·2^27: the result is 3·2^(3·2^27 + 27) - 1
E = 3 * 2 ** 27 + 27
print("final base = 3·2^E - 1 with E =", E)
steps_digits = math.floor(E * math.log10(2) + math.log10(3)) + 1
print("number of steps = final base - 2 = 3·2^E - 3, which has", steps_digits, "decimal digits")

The base at the end is $3\cdot2^{402653211}-1$; since the base starts at $2$ and grows by one per step, the sequence of $4$ takes $3\cdot2^{402653211}-3$ steps, a number with $121\,210\,695$ digits. Writing it down would take a few hundred thousand pages, and yet the ordinal argument proves that it is finite in three lines; it is a different way of counting, not a faster computer, that settles the question.

### P7.7

A hydra is a nested list: a node is the list of its children; a head is `[]`. We cut a random head; at step $n$ the hydra regrows $n$ copies.

In [ ]:
import copy

def hydra_cnf(t):
    # o(t) = sum of ω^{o(child)}, sorted non-increasingly; represented as sorted tuple of exponent CNFs (multiset)
    exps = sorted((hydra_cnf(c) for c in t), key=cmp_key, reverse=True)
    out = []
    for e in exps:
        if out and cnf_cmp(out[-1][0], e) == 0:
            out[-1] = (out[-1][0], out[-1][1] + 1)
        else:
            out.append((e, 1))
    return tuple(out)

from functools import cmp_to_key
cmp_key = cmp_to_key(cnf_cmp)

def heads(t, path=()):
    for i, c in enumerate(t):
        if not c:
            yield path + (i,)
        else:
            yield from heads(c, path + (i,))

def cut(t, path, n):
    t = copy.deepcopy(t)
    if len(path) == 1:
        del t[path[0]]
        return t
    grand = t
    for i in path[:-2]:
        grand = grand[i]
    parent = grand[path[-2]]
    del parent[path[-1]]
    for _ in range(n):
        grand.append(copy.deepcopy(parent))
    return t

def battle(t, rng, limit=10 ** 6):
    n, alpha = 1, hydra_cnf(t)
    while t:
        hs = list(heads(t))
        t = cut(t, rng.choice(hs), n)
        beta = hydra_cnf(t)
        assert cnf_cmp(beta, alpha) < 0
        alpha, n = beta, n + 1
        if n > limit:
            return None
    return n - 1

rng = random.Random(1)
for name, h in [("two necks with a head", [[[]], [[]]]),
                ("a neck with two heads", [[[], []]]),
                ("a neck with three heads", [[[], [], []]]),
                ("a path of length 3", [[[[]]]])]:
    lengths = [battle(h, rng) for _ in range(20)]
    print(f"{name:26s} ordinal {show(hydra_cnf(h)):24s} killed in {min(lengths)} to {max(lengths)} cuts")

Every battle ends, and the ordinal decreased at every one of the cuts (the `assert` would have stopped us otherwise). The survival time depends wildly on the order of the cuts. The hydra of Figure 7.2, with ordinal $\omega^{\omega^2+1}$, or a path of length $4$, with ordinal $\omega^{\omega^\omega}$, would outlast our patience and our memory, and Kirby and Paris proved that $\mathrm{PA}$ cannot prove that every hydra dies, although the ordinal argument above does it in a few lines.

### P7.8

In [ ]:
def zlen(bits):
    return 8 * len(zlib.compress(bytes(int(bits[i:i + 8], 2) for i in range(0, len(bits), 8)), 9))

n = 8000
random_bits = ''.join(random.choice('01') for _ in range(n))
periodic = '01101' * (n // 5)
squares = ''.join('1' if int(math.isqrt(k)) ** 2 == k else '0' for k in range(n))
champernowne = ''.join(bin(k)[2:] for k in range(1, 2000))[:n]
for name, s in [("random", random_bits), ("periodic 01101", periodic), ("indicator of squares", squares), ("Champernowne (binary)", champernowne)]:
    print(f"{name:24s} length {len(s)} bits, zlib: {zlen(s):5d} bits ({zlen(s) / len(s):.2f} per bit)")
# counting bound
for k in [1, 10, 20]:
    print(f"fraction of strings of length n compressible by at least {k} bits: < 2^-{k - 1} = {2 ** -(k - 1):.2e}")

Random strings do not compress (zlib even adds a little); periodic ones collapse; the indicator of squares, very sparse, compresses well. Champernowne's string, the binary numbers written one after the other, looks random to zlib but has a tiny Kolmogorov complexity: the program that writes it is two lines long. That is the gap Theorem 7.23 is about: an upper bound for $K$ is a program, and can be exhibited; a lower bound, ``this string has no short description'', is true of almost every string and can be proved for almost none.